# MXene Electrode Descriptor Extractor — multi-electrode

Upload a PDF, get one row per electrode reported in the paper.

**What changed from the single-electrode version:** a typical MXene supercapacitor paper
reports pristine Ti3C2Tx plus two or three modified variants (different etchant,
intercalant, annealing, composite loading). Extracting only the best one discards most
of the paper — and the *contrast* between variants is exactly the signal a descriptor
optimization model needs. This version returns an array of electrodes.

Descriptors are split into two levels:

- **Paper-level** (DOI, year, authors) — recorded once
- **Electrode-level** (18 descriptors) — repeated per electrode

**Cost:** roughly $0.05–0.12 per paper depending on how many electrodes it reports.
A cost readout prints after each run.

## Setup

In [1]:
%pip install --quiet openai pypdf pandas
print("done — restart the kernel if this was the first install")

Note: you may need to restart the kernel to use updated packages.
done — restart the kernel if this was the first install



[notice] A new release of pip is available: 23.2.1 -> 26.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
import os, getpass

if not os.environ.get("OPENAI_API_KEY"):
    os.environ["OPENAI_API_KEY"] = getpass.getpass("OpenAI API key: ")

# Or hardcode (don't commit the file if you do):
# os.environ["OPENAI_API_KEY"] = "sk-..."

print("key loaded:", bool(os.environ.get("OPENAI_API_KEY")))

key loaded: True


In [3]:
import json, re, time, pathlib, traceback
import pandas as pd
from pypdf import PdfReader
from openai import OpenAI

client = OpenAI()

# gpt-5.4        $2.50 / $15.00 per 1M tok  -- recommended
# gpt-5.6-terra  $2.50 / $15.00             -- newer family, comparable
# gpt-5.6-luna   $1.00 /  $6.00             -- ~60% cheaper, good for bulk
# gpt-5.4-mini   $0.75 /  $4.50             -- budget; validate before trusting
MODEL = "gpt-5.6-terra"

PRICING = {
    "gpt-5.4":       (2.50, 15.00),
    "gpt-5.6-terra": (2.50, 15.00),
    "gpt-5.6-sol":   (5.00, 30.00),
    "gpt-5.6-luna":  (1.00,  6.00),
    "gpt-5.4-mini":  (0.75,  4.50),
    "gpt-5.4-nano":  (0.20,  1.25),
}
_CODENAMES = {"sol", "terra", "luna", "mini", "nano"}


def model_tag(model=MODEL):
    """gpt-5.6-sol -> 'sol'; gpt-5.4 -> 'gpt-5-4' (no codename)."""
    tail = model.rsplit("-", 1)[-1]
    return tail if tail in _CODENAMES else model.replace(".", "-")


def output_dir(model=MODEL):
    """Per-model output root, created on demand. gpt-5.6-terra -> ./terra/

    Every artifact this notebook writes goes here, so switching MODEL never
    overwrites or silently merges into another model's results.
    """
    p = pathlib.Path(model_tag(model))
    p.mkdir(parents=True, exist_ok=True)
    return p


print("model:", MODEL, " -> output dir:", output_dir())

model: gpt-5.6-terra  -> output dir: terra


## What counts as a separate electrode

This is the decision that determines whether your dataset is clean, so it's worth
being explicit rather than leaving it to the model's judgment.

**Separate electrodes** — any sample with a distinct material identity that gets its
own performance number:

- different MXene formula (Ti3C2Tx vs V2CTx)
- different synthesis or etchant (HF vs MILD vs alkali)
- different intercalant or surface treatment
- different composite partner or loading ratio (MXene/rGO 1:1 vs 1:3)
- different annealing or post-treatment
- a control or reference sample the authors made themselves

**Not separate electrodes** — the same physical sample measured under varying
conditions:

- the same electrode at 5, 10, 50, 100 mV/s
- the same electrode at different current densities
- cycling data at timepoints
- three-electrode vs two-electrode cell configuration for the same material

That second list is the important one. If you let scan-rate sweeps become rows, a
single paper explodes into forty near-duplicate entries and your dataset is dominated
by whichever group ran the most scan rates. Each electrode gets **one** row carrying
its best reported capacitance plus the condition at which it was measured.

Literature values cited from *other* papers are excluded — those belong to the
original source, and pulling them in double-counts when you build a corpus.

In [4]:
# ---- Paper-level fields (recorded once) --------------------------------
PAPER_FIELDS = {
    "doi":     ("DOI string", "look for 'doi:' or 'https://doi.org/'"),
    "year":    ("Publication year, YYYY", "from header, footer, or citation block"),
    "authors": ("First author + et al. if more than 3", ""),
}

# ---- Electrode-level fields (repeated per electrode) -------------------
ELECTRODE_FIELDS = {
    "mxene_formula":     ("e.g. Ti3C2Tx, V2CTx, Mo2TiC2Tx", "normalize OCR-mangled subscripts"),
    "composition":       ("Transition metal + C/N layers + terminations (-OH, -F, =O); "
                          "include composite partner and ratio if present", ""),
    "synthesis_method":  ("HF etching, LiF/HCl (MILD), electrochemical, alkali-assisted, etc.", ""),
    "layer_no":          ("Layer count, or 'few-layer'/'multilayer'/'delaminated'", ""),
    "interlayer_spacing":("d-spacing from XRD (002) peak", "units A or nm"),
    "flake_size":        ("Lateral flake dimension from SEM/TEM/DLS", "units um or nm"),
    "porosity":          ("Porosity percentage", "from BET or mercury porosimetry"),
    "pore_diameter":     ("Average or median pore size", "units nm"),
    "tortuosity":        ("Tortuosity value, dimensionless", "often from impedance modeling"),
    "electrode_thickness":("Film or electrode thickness", "units um"),
    "mass_loading":      ("Active material per unit area", "units mg cm-2"),
    "electrolyte":       ("Full name and concentration, e.g. '3 M H2SO4'", ""),
    "ssa":               ("BET specific surface area", "units m2 g-1"),
    "scan_rate":         ("CV scan rate at which the reported capacitance was measured",
                          "units mV s-1"),
    "current_density":   ("GCD current density for the reported capacitance",
                          "units A g-1 or mA cm-2"),
    "gravimetric_capacitance": ("Cg / specific capacitance", "units F g-1"),
    "volumetric_capacitance":  ("Cv", "units F cm-3"),
    "areal_capacitance":       ("Ca", "units mF cm-2"),
}

def _field_obj(desc, hint):
    return {
        "type": "object",
        "additionalProperties": False,
        "required": ["value", "unit", "note", "confidence"],
        "properties": {
            "value": {"type": ["string", "null"],
                      "description": f"{desc}. {hint} Null if not reported for THIS electrode."},
            "unit":  {"type": ["string", "null"],
                      "description": "Unit exactly as reported. Do not convert."},
            "note":  {"type": "string",
                      "description": "If null, say where you looked and why it is absent. "
                                     "Otherwise note the measurement condition or source "
                                     "section. Under 20 words."},
            "confidence": {"type": "string", "enum": ["stated", "derived", "uncertain"],
                           "description": "stated = printed verbatim for this specific "
                                          "electrode; derived = computed or inherited from "
                                          "a shared methods section; uncertain = ambiguous "
                                          "which electrode the value belongs to."},
        },
    }

_paper_props = {k: _field_obj(d, h) for k, (d, h) in PAPER_FIELDS.items()}
_elec_props  = {k: _field_obj(d, h) for k, (d, h) in ELECTRODE_FIELDS.items()}

SCHEMA = {
    "type": "object",
    "additionalProperties": False,
    "required": ["paper", "electrodes", "n_electrodes", "summary"],
    "properties": {
        "paper": {
            "type": "object", "additionalProperties": False,
            "required": list(PAPER_FIELDS), "properties": _paper_props,
        },
        "n_electrodes": {
            "type": "integer",
            "description": "How many distinct electrodes you identified. Must equal the "
                           "length of the electrodes array.",
        },
        "electrodes": {
            "type": "array",
            "description": "One entry per distinct electrode material studied in this paper.",
            "items": {
                "type": "object", "additionalProperties": False,
                "required": ["label", "is_primary", "role", "fields"],
                "properties": {
                    "label": {"type": "string",
                              "description": "The authors' own name for this sample, verbatim "
                                             "where possible, e.g. 'Ti3C2Tx-MILD', 'MX/rGO-3', "
                                             "'pristine MXene'."},
                    "is_primary": {"type": "boolean",
                                   "description": "True for the single best-performing or "
                                                  "headline electrode of the paper."},
                    "role": {"type": "string",
                             "enum": ["primary", "variant", "control", "composite", "reference"],
                             "description": "primary = headline sample; variant = a synthesis "
                                            "or treatment variation; control = untreated or "
                                            "pristine baseline; composite = MXene combined with "
                                            "another material; reference = non-MXene comparison "
                                            "the authors themselves fabricated."},
                    "fields": {
                        "type": "object", "additionalProperties": False,
                        "required": list(ELECTRODE_FIELDS), "properties": _elec_props,
                    },
                },
            },
        },
        "summary": {
            "type": "string",
            "description": "3-5 sentences: MXene system and synthesis route, how many "
                           "electrodes were compared and how they differ, best capacitance "
                           "and its condition, key data gaps.",
        },
    },
}

print(f"schema: {len(PAPER_FIELDS)} paper-level + {len(ELECTRODE_FIELDS)} electrode-level fields")

schema: 3 paper-level + 18 electrode-level fields


## PDF text extraction

A note from testing this on real two-column ACS papers: the common advice to use
`pdftotext -layout` is wrong for this document class. Layout mode preserves horizontal
position, which interleaves the left and right columns line by line and shreds every
sentence. Reading-order extraction is what you want, and `pypdf` does it correctly.

In [5]:
def extract_pdf_text(path, verbose=True):
    """Extract full text in reading order. Returns (text, n_pages)."""
    reader = PdfReader(path)
    pages = [p.extract_text() or "" for p in reader.pages]
    text = "\n\n".join(pages)
    text = re.sub(r"[ \t]{3,}", " ", text)
    text = re.sub(r"\n{4,}", "\n\n", text)

    if verbose:
        chars = len(text)
        print(f"pages: {len(pages)}   chars: {chars:,}   est. tokens: ~{chars//4:,}")
        if chars / max(len(pages), 1) < 500:
            print("\n  WARNING: very little text per page — likely a scanned PDF.")
            print("  Run OCR first (ocrmypdf in.pdf out.pdf) before extracting.")
        print("\n--- first 400 chars ---")
        print(text[:400].strip())
    return text, len(pages)

print("ready")

ready


## The extraction call

The system prompt carries the electrode-splitting rules, because this is where the
model most needs constraining. Two instructions do the heavy lifting:

**Enumerate before extracting.** The model is told to first list the distinct samples,
then fill in each one. Without that, models tend to find two or three electrodes and
stop, missing variants introduced later in the results section.

**Inherit shared values, but flag them.** If the paper gives one electrolyte in the
methods and it applies to all samples, that value should propagate to every electrode
— marked `derived` rather than `stated`, so you can tell later which numbers were
measured per-sample and which were assumed. This is the main new failure mode in
multi-electrode extraction and the confidence flag is how you audit it.

In [6]:
SYSTEM_PROMPT = """You extract electrode characterization data from MXene research papers.

A paper usually reports SEVERAL distinct electrodes. Your job is to find ALL of them \
and return one entry per electrode.

WHAT COUNTS AS A SEPARATE ELECTRODE
Treat these as separate electrodes:
- different MXene formula (Ti3C2Tx vs V2CTx)
- different synthesis route or etchant (HF vs MILD vs alkali-assisted)
- different intercalant, surface treatment, or annealing condition
- different composite partner or loading ratio (MXene/rGO 1:1 vs 1:3)
- pristine or untreated control samples the authors fabricated
- non-MXene reference samples the authors fabricated themselves

Do NOT create separate entries for:
- the same electrode measured at different scan rates or current densities
- the same electrode at different cycle counts
- the same electrode in three-electrode vs two-electrode configuration
- values cited from OTHER published papers (comparison tables, literature benchmarks)

Each electrode gets exactly ONE entry, carrying its BEST reported capacitance and the \
condition at which that value was measured.

PROCEDURE
1. First read the whole paper and enumerate the distinct samples. Check the abstract, \
the experimental section, figure legends, and performance tables -- variants are often \
introduced late.
2. Then fill in the descriptors for each one.
3. Set n_electrodes to the number you found. It must match the array length.

EXTRACTION RULES
- Report only what the paper states. Never infer, estimate, or compute a value that is \
not printed. If a descriptor is absent for an electrode, set value to null and explain \
in the note.
- If a value is given once for the whole study (e.g. one electrolyte in the methods, \
one mass loading protocol) and clearly applies to every sample, propagate it to each \
electrode but set confidence to "derived" -- not "stated".
- Set confidence to "stated" only when the value is printed for THAT SPECIFIC electrode.
- Use "uncertain" when you cannot tell which electrode a reported number belongs to.
- Record units exactly as the authors write them. Do not convert.
- Use the authors' own sample names for label, verbatim where possible.
- Normalize OCR artifacts in formulas: "Ti3 C2 Tx" or "Ti3C2Tx" -> Ti3C2Tx.
- Exactly one electrode should have is_primary true: the headline or best performer."""


def extract_descriptors(text, model=MODEL, max_chars=140_000):
    """Single structured-output call. Returns (parsed_dict, usage_dict)."""
    if len(text) > max_chars:
        print(f"  note: truncating {len(text):,} -> {max_chars:,} chars")
        text = text[:max_chars]

    resp = client.chat.completions.create(
        model=model,
        messages=[
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user",
             "content": "Extract every distinct electrode reported in this paper.\n\n" + text},
        ],
        response_format={
            "type": "json_schema",
            "json_schema": {"name": "mxene_electrodes", "schema": SCHEMA, "strict": True},
        },
    )
    parsed = json.loads(resp.choices[0].message.content)
    u = resp.usage
    return parsed, {"in": u.prompt_tokens, "out": u.completion_tokens, "model": model}


def report_cost(usage, n_elec=None):
    rin, rout = PRICING.get(usage["model"], (2.50, 15.00))
    cost = usage["in"] / 1e6 * rin + usage["out"] / 1e6 * rout
    extra = f"   ({n_elec} electrodes, ${cost/max(n_elec,1):.4f} each)" if n_elec else ""
    print(f"tokens  in={usage['in']:,}  out={usage['out']:,}   "
          f"cost ~${cost:.4f}   [{usage['model']}]{extra}")
    return cost

print("ready")

ready


## Consistency checks

Structured outputs guarantee the *shape* of the response, not its correctness. These
checks catch the things the schema can't: a miscounted `n_electrodes`, duplicate labels
(usually a sign the model split one sample into two), no primary or several primaries,
and electrodes where almost nothing was populated.

The last check is the most useful in practice. An electrode row with two of eighteen
fields filled usually means the model picked up a passing mention rather than a real
characterized sample.

In [7]:
def validate(parsed, verbose=True):
    """Return a list of warning strings. Empty list means everything looks consistent."""
    warns = []
    elecs = parsed["electrodes"]

    if parsed.get("n_electrodes") != len(elecs):
        warns.append(f"n_electrodes={parsed.get('n_electrodes')} but array has {len(elecs)}")

    if not elecs:
        warns.append("no electrodes extracted at all")
        return warns

    labels = [e["label"].strip().lower() for e in elecs]
    dupes = {l for l in labels if labels.count(l) > 1}
    if dupes:
        warns.append(f"duplicate labels: {sorted(dupes)}")

    n_prim = sum(1 for e in elecs if e.get("is_primary"))
    if n_prim == 0:
        warns.append("no electrode marked primary")
    elif n_prim > 1:
        warns.append(f"{n_prim} electrodes marked primary (expected 1)")

    for e in elecs:
        f = e["fields"]
        filled = sum(1 for v in f.values() if v["value"] is not None)
        if filled <= 2:
            warns.append(f"'{e['label']}' has only {filled}/{len(f)} fields — "
                         f"may be a spurious entry")
        caps = [f[k]["value"] for k in
                ("gravimetric_capacitance", "volumetric_capacitance", "areal_capacitance")]
        if not any(caps):
            warns.append(f"'{e['label']}' has no capacitance of any kind")

    if verbose:
        if warns:
            print("CHECKS — review these:")
            for w in warns:
                print("  ! " + w)
        else:
            print("checks passed")
    return warns

print("ready")

ready


## Formatting

Two views of the same data.

`to_long_dataframe` gives one row per electrode with descriptors as columns — this is
what you want for analysis and what stacks cleanly across a corpus.

`show_electrode` gives the original field-by-field table for a single electrode, which
is easier to eyeball when you're hand-validating.

In [8]:
PAPER_LABELS = {"doi": "DOI", "year": "Year", "authors": "Authors"}
ELEC_LABELS = {
    "mxene_formula": "MXene Formula", "composition": "Composition",
    "synthesis_method": "Synthesis Method", "layer_no": "Layer No",
    "interlayer_spacing": "Interlayer Spacing (d)", "flake_size": "Flake Size",
    "porosity": "Porosity", "pore_diameter": "Pore Diameter", "tortuosity": "Tortuosity",
    "electrode_thickness": "Electrode Thickness", "mass_loading": "Mass Loading",
    "electrolyte": "Electrolyte", "ssa": "SSA", "scan_rate": "Scan Rate",
    "current_density": "Current Density",
    "gravimetric_capacitance": "Gravimetric Capacitance",
    "volumetric_capacitance": "Volumetric Capacitance",
    "areal_capacitance": "Areal Capacitance",
}

def to_long_dataframe(parsed, source_file=""):
    """One row per electrode. Values, units and confidence as separate columns."""
    paper = {k: parsed["paper"][k]["value"] for k in PAPER_LABELS}
    rows = []
    for e in parsed["electrodes"]:
        row = {"source_file": source_file, **paper,
               "electrode_label": e["label"], "role": e["role"],
               "is_primary": e["is_primary"]}
        for k in ELEC_LABELS:
            f = e["fields"][k]
            row[k] = f["value"]
            row[k + "__unit"] = f["unit"]
            row[k + "__conf"] = f["confidence"]
        rows.append(row)
    return pd.DataFrame(rows)


def show_electrode(parsed, idx=0):
    """Field-by-field table for one electrode — useful for hand-validation."""
    e = parsed["electrodes"][idx]
    rows = [{"Field": lab,
             "Value": e["fields"][k]["value"] or "Not reported",
             "Unit": e["fields"][k]["unit"] or "—",
             "Conf": e["fields"][k]["confidence"],
             "Notes": e["fields"][k]["note"]}
            for k, lab in ELEC_LABELS.items()]
    df = pd.DataFrame(rows)
    print(f"\n[{idx}] {e['label']}   role={e['role']}"
          f"{'   (PRIMARY)' if e['is_primary'] else ''}")
    try:
        from IPython.display import display
        display(df.style.hide(axis="index"))
    except Exception:
        print(df.to_string(index=False))
    return df


def show_overview(parsed):
    """Compact comparison across electrodes — the view you actually want first."""
    rows = []
    for i, e in enumerate(parsed["electrodes"]):
        f = e["fields"]
        filled = sum(1 for v in f.values() if v["value"] is not None)
        cap = f["gravimetric_capacitance"]
        rows.append({
            "#": i,
            "Label": e["label"],
            "Role": e["role"],
            "Formula": f["mxene_formula"]["value"] or "—",
            "Synthesis": (f["synthesis_method"]["value"] or "—")[:28],
            "Cg": f"{cap['value']} {cap['unit'] or ''}".strip() if cap["value"] else "—",
            "Condition": f["current_density"]["value"] or f["scan_rate"]["value"] or "—",
            "Filled": f"{filled}/{len(f)}",
        })
    df = pd.DataFrame(rows)
    try:
        from IPython.display import display, Markdown
        display(df.style.hide(axis="index"))
        display(Markdown(f"**Summary**\n\n{parsed['summary']}"))
    except Exception:
        print(df.to_string(index=False))
        print("\nSummary:", parsed["summary"])
    return df

print("ready")

ready


---

# Step 1 — Point at your PDF

In [10]:
PDF_PATH = "./H2SO4/1-s2.0-S001346862101762X-main.pdf"   # <-- change this

# Colab: uncomment for an upload widget
# from google.colab import files
# up = files.upload(); PDF_PATH = list(up.keys())[0]

assert pathlib.Path(PDF_PATH).exists(), f"not found: {PDF_PATH}"
text, n_pages = extract_pdf_text(PDF_PATH)

pages: 7   chars: 41,652   est. tokens: ~10,413

--- first 400 chars ---
Electrochimica Acta 401 (2022) 139476 
Contents lists available at ScienceDirect 
Electrochimica Acta 
journal homepage: www.elsevier.com/locate/electacta 
High capacitance of MXene (Ti 3 C 2 T x ) through Intercalation and Surface 
Modiﬁcation in Molten Salt 
Liang Guo a , Wei-Yan Jiang b , c , d , Miao Shen b , c , d , ∗, Cong Xu a , Chen-Xu Ding a , 
Su-Fang Zhao b , c , d , Tao-Tao Yuan a , Ch


# Step 2 — Extract

In [11]:
stem = pathlib.Path(PDF_PATH).stem
cache_json = output_dir() / f"{stem}_electrodes.json"

if cache_json.exists():
    # Already extracted — reuse the saved result instead of paying to re-run.
    rec = json.loads(cache_json.read_text())
    parsed = {k: rec[k] for k in ("paper", "electrodes", "n_electrodes", "summary")}
    usage = {"in": 0, "out": 0, "model": rec.get("model", MODEL)}
    cost = rec.get("cost_usd", 0.0)
    warns = rec.get("warnings", validate(parsed, verbose=False))
    print(f"cached — loaded {cache_json.name} (no API call)")
    print(f"found {len(parsed['electrodes'])} electrodes   prior cost ~${cost:.4f}")
    if warns:
        print()
        for w in warns:
            print("  ! " + w)
else:
    parsed, usage = extract_descriptors(text, model=MODEL)

    print(f"\nfound {len(parsed['electrodes'])} electrodes")
    cost = report_cost(usage, n_elec=len(parsed["electrodes"]))
    print()
    warns = validate(parsed)


found 3 electrodes
tokens  in=17,518  out=2,429   cost ~$0.0802   [gpt-5.6-terra]   (3 electrodes, $0.0267 each)

CHECKS — review these:
  ! 'M-Ti3C2Tx' has no capacitance of any kind


# Step 3 — Look at the results

Overview first, then drill into any electrode that looks off.

In [12]:
_ = show_overview(parsed)

#,Label,Role,Formula,Synthesis,Cg,Condition,Filled
0,HF-etched Ti3C2Tx,control,Ti3C2Tx,HF etching,60 F g−1,1,8/18
1,M-Ti3C2Tx,variant,Ti3C2Tx,LiCl-KCl molten-salt treatme,—,—,6/18
2,KM-Ti3C2Tx,primary,Ti3C2Tx,LiCl-KCl-K2CO3 molten-salt t,340 F g−1,0.5,8/18


**Summary**

This study compares three Ti3C2Tx electrodes: HF-etched Ti3C2Tx, LiCl-KCl molten-salt-treated M-Ti3C2Tx, and LiCl-KCl-K2CO3 molten-salt-treated KM-Ti3C2Tx. The molten-salt carbonate treatment introduces potassium oxygenated-complex intercalation, increases O termination content, and expands the layer spacing to 1.05–1.21 nm. KM-Ti3C2Tx is the headline and best-performing electrode, reaching 340 F g−1 at 0.5 A g−1 in 1 M H2SO4; the abstract/conclusion additionally report 323.6 F g−1 at 1 A g−1. Electrode thickness, flake size, BET surface area, pore diameter, porosity, tortuosity, and volumetric/areal capacitances were not reported.

In [13]:
# Full descriptor table for one electrode — change the index to inspect others
_ = show_electrode(parsed, 0)

# for i in range(len(parsed["electrodes"])):
#     show_electrode(parsed, i)


[0] HF-etched Ti3C2Tx   role=control


Field,Value,Unit,Conf,Notes
MXene Formula,Ti3C2Tx,—,stated,Prepared by HF etching Ti3AlC2.
Composition,"Ti3C2Tx (F-, O-, and OH-terminated)",—,stated,"HF-etched material with F, O and OH terminations discussed."
Synthesis Method,HF etching,—,stated,"Selective HF etching: 30 wt% HF, 18 h at room temperature."
Layer No,Not reported,—,stated,Layered structure stated; layer count not reported.
Interlayer Spacing (d),approximately 0.96,nm,stated,HRTEM layer spacing.
Flake Size,Not reported,—,stated,No lateral flake dimension reported.
Porosity,Not reported,—,stated,No porosity percentage reported.
Pore Diameter,Not reported,—,stated,No pore diameter reported.
Tortuosity,Not reported,—,stated,No tortuosity value reported.
Electrode Thickness,Not reported,—,stated,No electrode thickness reported.


# Step 4 — Save

CSV is one row per electrode, ready to stack across papers. JSON keeps the nested
structure plus notes and warnings.

In [14]:
stem = pathlib.Path(PDF_PATH).stem
OUT = output_dir()

df = to_long_dataframe(parsed, source_file=PDF_PATH)
df.to_csv(OUT / f"{stem}_electrodes.csv", index=False)

with open(OUT / f"{stem}_electrodes.json", "w") as fh:
    json.dump({"source_file": PDF_PATH, "model": usage["model"],
               "cost_usd": round(cost, 5), "warnings": warns, **parsed}, fh, indent=2)

print(f"wrote {OUT}/{stem}_electrodes.csv  ({len(df)} rows)")
print(f"wrote {OUT}/{stem}_electrodes.json")
df.head()

wrote terra/1-s2.0-S001346862101762X-main_electrodes.csv  (3 rows)
wrote terra/1-s2.0-S001346862101762X-main_electrodes.json


,source_file,doi,year,authors,electrode_label,role,is_primary,mxene_formula,mxene_formula__unit,mxene_formula__conf,...,current_density__conf,gravimetric_capacitance,gravimetric_capacitance__unit,gravimetric_capacitance__conf,volumetric_capacitance,volumetric_capacitance__unit,volumetric_capacitance__conf,areal_capacitance,areal_capacitance__unit,areal_capacitance__conf
0,./H2SO4/1-s2.0-S001346862101762X-main.pdf,10.1016/j.electacta.2021.139476,2022,Guo et al.,HF-etched Ti3C2Tx,control,False,Ti3C2Tx,None,stated,...,stated,60,F g−1,stated,None,None,stated,None,None,stated
1,./H2SO4/1-s2.0-S001346862101762X-main.pdf,10.1016/j.electacta.2021.139476,2022,Guo et al.,M-Ti3C2Tx,variant,False,Ti3C2Tx,None,stated,...,stated,None,None,stated,None,None,stated,None,None,stated
2,./H2SO4/1-s2.0-S001346862101762X-main.pdf,10.1016/j.electacta.2021.139476,2022,Guo et al.,KM-Ti3C2Tx,primary,True,Ti3C2Tx,None,stated,...,stated,340,F g−1,stated,None,None,stated,None,None,stated


---

# Batch mode

Writes each result to JSON as it completes and skips finished files on rerun, so a
crash at paper 340 doesn't cost you the first 339. Failures are collected and reported
rather than halting the run.

Papers whose consistency checks fired are listed separately at the end — those are the
ones to hand-inspect first.

**Before running this on a real corpus:** hand-check 10–20 papers. With multi-electrode
extraction the failure that will hurt you is not a crash, it is the model attributing
electrode B's capacitance to electrode A on a subset of papers. That produces a dataset
that looks entirely reasonable and regresses to nonsense.

In [15]:
def estimate_cost(folder, model=MODEL, recursive=True, avg_electrodes=3.0):
    """Dry run: count PDFs, measure their real token size, project the bill.

    Reads the PDFs locally (free) and estimates output from avg_electrodes.
    Run this before spending money on a corpus."""
    folder = pathlib.Path(folder)
    pdfs = sorted(folder.rglob("*.pdf") if recursive else folder.glob("*.pdf"))
    if not pdfs:
        print(f"no PDFs found in {folder}")
        return

    total_in, sizes, unreadable = 0, [], []
    for p in pdfs:
        try:
            txt, _ = extract_pdf_text(p, verbose=False)
            if len(txt.strip()) < 1000:
                unreadable.append(p.name)
            tok = len(txt) // 4
            sizes.append(tok)
            total_in += tok
        except Exception as e:
            unreadable.append(f"{p.name} ({type(e).__name__})")

    # ~350 output tokens per electrode + ~200 for paper block and summary
    out_per_paper = 200 + 350 * avg_electrodes
    total_out = out_per_paper * len(sizes)
    prompt_overhead = 1400 * len(sizes)   # system prompt, sent every call
    total_in += prompt_overhead

    print(f"{len(pdfs)} PDFs in {folder}")
    if sizes:
        print(f"input tokens: {total_in:,}  (median paper {sorted(sizes)[len(sizes)//2]:,})")
    print(f"projected output: ~{int(total_out):,} tokens "
          f"(assuming {avg_electrodes} electrodes/paper)")
    if unreadable:
        print(f"\n{len(unreadable)} PDFs with little or no text layer — need OCR:")
        for u in unreadable[:10]:
            print("   " + u)
        if len(unreadable) > 10:
            print(f"   ... and {len(unreadable)-10} more")

    print("\nprojected cost:")
    for m, (rin, rout) in PRICING.items():
        c = total_in / 1e6 * rin + total_out / 1e6 * rout
        star = "  <-- current" if m == model else ""
        print(f"   {m:16s} ${c:7.2f}   (batch API: ${c/2:6.2f}){star}")
    return {"n_pdfs": len(pdfs), "in": total_in, "out": int(total_out),
            "unreadable": unreadable}


def batch_extract(folder, out_dir=None, model=MODEL, sleep=0.5,
                  recursive=True, limit=None):
    """Extract every PDF in a folder. Resumable: reruns skip completed files.

    limit=N processes only the first N unprocessed papers — use it to sanity-check
    a handful before committing to the full corpus."""
    folder = pathlib.Path(folder)
    # Resolved from the model actually passed in, not a def-time constant, so
    # the resume cache is per-model and a MODEL switch cannot merge corpora.
    out_dir = pathlib.Path(out_dir) if out_dir else output_dir(model) / "extracted"
    out_dir.mkdir(parents=True, exist_ok=True)

    pdfs = sorted(folder.rglob("*.pdf") if recursive else folder.glob("*.pdf"))
    print(f"{len(pdfs)} PDFs found in {folder}  ->  {out_dir}  [{model}]\n")

    results, failures, flagged = [], [], []
    total_cost, total_elec, n_done = 0.0, 0, 0

    for i, pdf in enumerate(pdfs, 1):
        # key cache on path relative to folder, so subdirectories with
        # same-named files do not collide
        rel = pdf.relative_to(folder)
        safe = str(rel.with_suffix("")).replace("/", "__").replace("\\", "__")
        out_json = out_dir / f"{safe}.json"

        if out_json.exists():
            rec = json.loads(out_json.read_text())
            results.append(rec)
            total_elec += len(rec.get("electrodes", []))
            print(f"[{i}/{len(pdfs)}] {rel} — cached")
            continue

        if limit is not None and n_done >= limit:
            continue

        print(f"[{i}/{len(pdfs)}] {rel}", end="  ")
        try:
            txt, _ = extract_pdf_text(pdf, verbose=False)
            if len(txt.strip()) < 1000:
                raise ValueError("almost no text layer — likely scanned, needs OCR")

            p, u = extract_descriptors(txt, model=model)
            rin, rout = PRICING.get(model, (2.50, 15.00))
            cst = u["in"] / 1e6 * rin + u["out"] / 1e6 * rout
            total_cost += cst
            n_done += 1

            w = validate(p, verbose=False)
            rec = {"source_file": pdf.name, "source_path": str(rel),
                   "model": model, "cost_usd": round(cst, 5), "warnings": w, **p}
            out_json.write_text(json.dumps(rec, indent=2))
            results.append(rec)

            n = len(p["electrodes"])
            total_elec += n
            print(f"ok — {n} electrodes, ${cst:.4f}" + (f", {len(w)} warnings" if w else ""))
            if w:
                flagged.append((pdf.name, w))

        except KeyboardInterrupt:
            print("\n\ninterrupted — progress is saved, rerun to resume")
            break
        except Exception as e:
            print(f"FAILED — {type(e).__name__}: {e}")
            failures.append((pdf.name, str(e)))
        time.sleep(sleep)

    print("\n" + "=" * 62)
    print(f"{len(results)} papers, {total_elec} electrodes total")
    print(f"{len(failures)} failed, {len(flagged)} flagged for review")
    print(f"cost this run: ${total_cost:.2f}")
    if failures:
        print("\nfailures:")
        for n_, e_ in failures:
            print(f"   {n_}: {e_}")
    if flagged:
        print("\nflagged (hand-check these first):")
        for n_, ws in flagged:
            print(f"   {n_}: {'; '.join(ws[:2])}")
    return results, failures, flagged


def corpus_dataframe(results):
    """Stack every electrode from every paper. source_file is the first column."""
    frames = []
    for r in results:
        if not r.get("electrodes"):
            continue
        d = to_long_dataframe(r, source_file=r.get("source_file", ""))
        if "source_path" in r:
            d.insert(1, "source_path", r["source_path"])
        frames.append(d)
    return pd.concat(frames, ignore_index=True) if frames else pd.DataFrame()

print("ready")

ready


In [16]:
# ---- 1. Preflight: what will this cost? (reads PDFs locally, no API calls) ----
FOLDER = "./H2SO4"          # <-- your folder

est = estimate_cost(FOLDER)

46 PDFs in H2SO4
input tokens: 584,436  (median paper 11,341)
projected output: ~57,500 tokens (assuming 3.0 electrodes/paper)

projected cost:
   gpt-5.4          $   2.32   (batch API: $  1.16)
   gpt-5.6-terra    $   2.32   (batch API: $  1.16)  <-- current
   gpt-5.6-sol      $   4.65   (batch API: $  2.32)
   gpt-5.6-luna     $   0.93   (batch API: $  0.46)
   gpt-5.4-mini     $   0.70   (batch API: $  0.35)
   gpt-5.4-nano     $   0.19   (batch API: $  0.09)


### 2. Trial run on a few papers

Before committing to the whole folder, do five. Read the output, check a couple of
electrodes against the actual PDFs, and confirm the electrode splitting matches what
you'd have done by hand. This costs cents and is the highest-value step in the process.

Results are cached, so nothing here is repeated work — the full run picks up where
this leaves off.

In [17]:
results, failures, flagged = batch_extract(FOLDER, limit=5)

corpus = corpus_dataframe(results)
print(f"\n{len(corpus)} electrodes from {corpus['source_file'].nunique()} papers")
corpus[["source_file", "electrode_label", "role", "mxene_formula",
        "gravimetric_capacitance", "electrolyte"]]

46 PDFs found in H2SO4  ->  terra\extracted  [gpt-5.6-terra]

[1/46] 1-s2.0-S0008622325000375-main.pdf  ok — 3 electrodes, $0.0924
[2/46] 1-s2.0-S001346862101762X-main.pdf  ok — 3 electrodes, $0.0806, 1 warnings
[3/46] 1-s2.0-S0013468622000433-main.pdf  ok — 4 electrodes, $0.1000, 3 warnings
[4/46] 1-s2.0-S0144861722014242-main.pdf  ok — 4 electrodes, $0.1008, 1 warnings
[5/46] 1-s2.0-S0169433219330661-main.pdf  ok — 1 electrodes, $0.0621

5 papers, 15 electrodes total
0 failed, 3 flagged for review
cost this run: $0.44

flagged (hand-check these first):
   1-s2.0-S001346862101762X-main.pdf: 'M-Ti3C2Tx' has no capacitance of any kind
   1-s2.0-S0013468622000433-main.pdf: 'MP0' has no capacitance of any kind; 'MP2' has no capacitance of any kind
   1-s2.0-S0144861722014242-main.pdf: 'CNF/PANI' has no capacitance of any kind

15 electrodes from 5 papers


,source_file,electrode_label,role,mxene_formula,gravimetric_capacitance,electrolyte
0,1-s2.0-S0008622325000375-main.pdf,NS-Ti3C2Tx-P,primary,Ti3C2Tx,424,2 M H2SO4
1,1-s2.0-S0008622325000375-main.pdf,NS-Ti3C2Tx-A,variant,Ti3C2Tx,358,2 M H2SO4
2,1-s2.0-S0008622325000375-main.pdf,Ti3C2Tx,control,Ti3C2Tx,330,2 M H2SO4
3,1-s2.0-S001346862101762X-main.pdf,HF-etched Ti3C2Tx,control,Ti3C2Tx,60,1 M H2SO4 solution
4,1-s2.0-S001346862101762X-main.pdf,M-Ti3C2Tx,variant,Ti3C2Tx,None,1 M H2SO4 solution
5,1-s2.0-S001346862101762X-main.pdf,KM-Ti3C2Tx,primary,Ti3C2Tx,340,1 M H2SO4 solution
6,1-s2.0-S0013468622000433-main.pdf,MP0,control,Ti3C2Tx,None,1 M H2SO4 aqueous solution
7,1-s2.0-S0013468622000433-main.pdf,MP2,composite,Ti3C2Tx,None,1 M H2SO4 aqueous solution
8,1-s2.0-S0013468622000433-main.pdf,MP5,primary,Ti3C2Tx,425.7,1 M H2SO4 aqueous solution
9,1-s2.0-S0013468622000433-main.pdf,MP8,composite,Ti3C2Tx,None,1 M H2SO4 aqueous solution


### 3. Full run

Remove `limit` to process everything. Already-extracted papers are skipped, so this
resumes rather than restarting. Safe to interrupt with the stop button — progress is
written to disk after every paper.

In [26]:
results, failures, flagged = batch_extract(FOLDER)

corpus = corpus_dataframe(results)
corpus_csv = output_dir() / "h2so4_corpus.csv"
corpus.to_csv(corpus_csv, index=False)
print(f"\nwrote {corpus_csv} — {len(corpus)} electrodes "
      f"from {corpus['source_file'].nunique()} papers")

78 PDFs found in H2SO4  ->  terra\extracted  [gpt-5.6-terra]

[1/78] 1-s2.0-S0008622325000375-main.pdf — cached
[2/78] 1-s2.0-S001346862101762X-main.pdf — cached
[3/78] 1-s2.0-S0013468622000433-main.pdf — cached
[4/78] 1-s2.0-S0144861722014242-main.pdf — cached
[5/78] 1-s2.0-S0169433219330661-main.pdf — cached
[6/78] 1-s2.0-S0272884220315613-main.pdf — cached
[7/78] 1-s2.0-S0925838821027134-main.pdf — cached
[8/78] 1-s2.0-S0925838822030389-main.pdf — cached
[9/78] 1-s2.0-S0925838823046583-main.pdf — cached
[10/78] 1-s2.0-S1385894724087011-main.pdf — cached
[11/78] 1-s2.0-S1388248114002896-main.pdf — cached
[12/78] 1-s2.0-S2211285517303634-main.pdf — cached
[13/78] 1-s2.0-S2352940719303622-main.pdf — cached
[14/78] 1-s2.0-S2369969821000785-main.pdf  ok — 2 electrodes, $0.0700, 1 warnings
[15/78] 1-s2.0-S2405829723005251-main.pdf  ok — 9 electrodes, $0.1648, 4 warnings
[16/78] 10.1002@batt.201800014.pdf — cached
[17/78] 1399231.pdf  ok — 4 electrodes, $0.1124
[18/78] 201807260933092014.p

incorrect startxref pointer(4)
parsing for Object Streams
Object 647 0 not defined.


FAILED — PdfReadError: Invalid object in /Pages
[60/78] s40820-023-01204-4.pdf — cached
[61/78] s40820-024-01567-2.pdf  ok — 2 electrodes, $0.0916
[62/78] s41467-019-09398-1.pdf — cached
[63/78] s41586-018-0109-z.pdf — cached
[64/78] Single-Step-Synthesis-of-Exfoliated-Ti-sub-3-sub-C-sub-2-sub-T-sub-x-sub-MXene-through-NaBF.pdf — cached
[65/78] Small - 2023 - Xu - Boron Quantum Dots Pillared Ti3C2Tx Membrane Electrode with High Rate Performance for Supercapacitor.pdf — cached
[66/78] Small Methods - 2025 - Kouchi - StableTi3C2Tx MXene Ink Formulation and High‐Resolution Aerosol Jet Printing for.pdf  ok — 5 electrodes, $0.1336, 1 warnings
[67/78] Small Science - 2025 - Li - Operando X‐Ray Diffraction Study of MXene Electrode Structure in Supercapacitors with Alkali.pdf  ok — 1 electrodes, $0.0997
[68/78] Small_2022_Guo_Thickness80_90Independent_Capacitive_Performance_of_Holey_Ti3C2Tx_Film_Prepared_through_a_Mild_Oxidation.pdf  ok — 6 electrodes, $0.1374, 3 warnings
[69/78] SmartMat - 20

### 3b. Rule-based filter (Ti3C2 + LiF/HCl + H2SO4)

Same deterministic checks as `scripts/filter_mxene.py`, ported to run on the corpus dataframe. Each electrode is labelled PASS / FAIL / AMBIGUOUS:

- **Formula** — primary formula (before the first `;`, `/`, `,`, or `(secondary)`) contains Ti3C2
- **Synthesis** — mentions both LiF and HCl
- **Electrolyte** — mentions H2SO4 / sulfuric acid

AMBIGUOUS means one of those fields was empty (or a "not reported" placeholder), so the row couldn't be judged rather than genuinely failing. Reasoning columns are kept so every verdict is auditable.

In [27]:
import unicodedata

# ---- ported verbatim from scripts/filter_mxene.py --------------------------
_SUBSCRIPT_MAP = str.maketrans("\u2080\u2081\u2082\u2083\u2084\u2085\u2086\u2087\u2088\u2089\u2093", "0123456789x")

_NULL_PATTERN = re.compile(
    r"not\s+reported|not\s+available|not\s+specified|not\s+applicable"
    r"|\bn/a\b|\bna\b|\bnone\b|\bunknown\b|\bunspecified\b",
    re.IGNORECASE,
)


def _normalise(s: str) -> str:
    """NFKC normalisation + subscript collapse, lowercased."""
    return unicodedata.normalize("NFKC", s).translate(_SUBSCRIPT_MAP).lower()


def check_formula(formula: str):
    norm = _normalise(formula)
    primary = re.split(r";|/|,|\(secondary\)|\(minor\)|\(second", norm)[0].strip()
    ti3c2_pat = re.compile(r"ti3c2")
    if ti3c2_pat.search(primary):
        return True, f"Primary formula '{formula.strip()}' contains Ti3C2 (Ti3C2Tx or Ti3C2)."
    if ti3c2_pat.search(norm):
        return False, f"Ti3C2 appears in '{formula.strip()}' but is not the primary formula."
    if not norm.strip():
        return None, "Formula cell is empty."
    return False, f"Primary formula '{formula.strip()}' is not Ti3C2Tx or Ti3C2."


def check_electrolyte(electrolyte: str):
    if not electrolyte.strip():
        return None, "Electrolyte cell is empty."
    norm = _normalise(electrolyte)
    h2so4_pat = re.compile(r"h2so4|su[lp]phuric[\s\-]?acid", re.IGNORECASE)
    if h2so4_pat.search(norm):
        return True, "Electrolyte explicitly mentions H2SO4 (sulfuric acid)."
    return False, f"Electrolyte '{electrolyte.strip()}' is not H2SO4."


def check_synthesis(synthesis: str):
    if not synthesis.strip():
        return None, "Synthesis cell is empty."
    norm = _normalise(synthesis)
    lif_pat = re.compile(r"\blif\b|lithium[\s\-]?fluoride")
    hcl_pat = re.compile(r"\bhcl\b|hydrochloric[\s\-]?acid")
    has_lif = bool(lif_pat.search(norm))
    has_hcl = bool(hcl_pat.search(norm))
    if has_lif and has_hcl:
        return True, "Synthesis explicitly mentions both LiF and HCl."
    if not has_lif and not has_hcl:
        return False, "Synthesis mentions neither LiF nor HCl."
    if not has_lif:
        return False, "Synthesis mentions HCl but not LiF."
    return False, "Synthesis mentions LiF but not HCl."


def evaluate_row(formula: str, synthesis: str, electrolyte: str) -> dict:
    fm, fr = check_formula(formula)
    sm, sr = check_synthesis(synthesis)
    em, er = check_electrolyte(electrolyte)

    if fm is None or sm is None or em is None:
        overall = "AMBIGUOUS"
    elif fm and sm and em:
        overall = "PASS"
    else:
        overall = "FAIL"

    parts = []
    if fm is True and sm is True and em is True:
        parts.append("Passes all criteria.")
    else:
        if fm is False:
            parts.append("Formula does not match.")
        if sm is False:
            parts.append("Synthesis does not match.")
        if em is False:
            parts.append("Electrolyte does not match.")
        if fm is None or sm is None or em is None:
            parts.append("One or more fields are empty.")

    return {
        "formula_match": fm, "formula_reasoning": fr,
        "synthesis_match": sm, "synthesis_reasoning": sr,
        "electrolyte_match": em, "electrolyte_reasoning": er,
        "overall": overall,
        "overall_reasoning": " ".join(parts) or overall,
    }


def _scrub(v):
    """Blank out 'not reported' style placeholders, matching the script."""
    if not isinstance(v, str):
        return ""
    return "" if _NULL_PATTERN.search(v) else v


def filter_corpus(df):
    """Return df with PASS/FAIL/AMBIGUOUS verdict + reasoning columns appended.

    Reads the notebook's long-format columns (mxene_formula, synthesis_method,
    electrolyte) rather than the script's title-case names.
    """
    verdicts = []
    for _, row in df.iterrows():
        formula = _scrub(str(row.get("mxene_formula", "") or "")).strip()
        synth = _scrub(str(row.get("synthesis_method", "") or "")).strip()
        elec = _scrub(str(row.get("electrolyte", "") or "")).strip()
        verdicts.append(evaluate_row(formula, synth, elec))
    verdict_df = pd.DataFrame(verdicts, index=df.index)
    return pd.concat([df, verdict_df], axis=1)


# ---- run it on the corpus --------------------------------------------------
corpus_filtered = filter_corpus(corpus)

counts = corpus_filtered["overall"].value_counts()
passed     = corpus_filtered[corpus_filtered["overall"] == "PASS"]
failed     = corpus_filtered[corpus_filtered["overall"] == "FAIL"]
ambiguous  = corpus_filtered[corpus_filtered["overall"] == "AMBIGUOUS"]

print(f"{len(corpus_filtered)} electrodes filtered")
print(f"  PASS      : {len(passed):>4}")
print(f"  FAIL      : {len(failed):>4}")
print(f"  AMBIGUOUS : {len(ambiguous):>4}")

filtered_csv = output_dir() / "h2so4_corpus_filtered.csv"
corpus_filtered.to_csv(filtered_csv, index=False)
print(f"\nwrote {filtered_csv} (all rows, with verdict columns)")

passed[["source_file", "electrode_label", "mxene_formula",
        "synthesis_method", "electrolyte", "gravimetric_capacitance"]]

366 electrodes filtered
  PASS      :  192
  FAIL      :  130
  AMBIGUOUS :   44

wrote terra\h2so4_corpus_filtered.csv (all rows, with verdict columns)


,source_file,electrode_label,mxene_formula,synthesis_method,electrolyte,gravimetric_capacitance
0,1-s2.0-S0008622325000375-main.pdf,NS-Ti3C2Tx-P,Ti3C2Tx,LiF/HCl etching; photothermal-assisted thioure...,2 M H2SO4,424
1,1-s2.0-S0008622325000375-main.pdf,NS-Ti3C2Tx-A,Ti3C2Tx,LiF/HCl etching; thiourea N/S co-doping by ann...,2 M H2SO4,358
2,1-s2.0-S0008622325000375-main.pdf,Ti3C2Tx,Ti3C2Tx,LiF/HCl etching,2 M H2SO4,330
6,1-s2.0-S0013468622000433-main.pdf,MP0,Ti3C2Tx,LiF/HCl etching followed by delamination and v...,1 M H2SO4 aqueous solution,None
7,1-s2.0-S0013468622000433-main.pdf,MP2,Ti3C2Tx,"LiF/HCl etching, PANI physical mixing, and vac...",1 M H2SO4 aqueous solution,None
...,...,...,...,...,...,...
353,zhang2018.pdf,I-Ti3C2Tx,Ti3C2Tx,LiF/HCl minimally intensive layer delamination...,PVA/H2SO4 gel electrolyte,None
354,zhang2018.pdf,Y-Ti3C2Tx,Ti3C2Tx,LiF/HCl minimally intensive layer delamination...,PVA/H2SO4 gel electrolyte,None
356,zhang2018.pdf,S-Ti3C2Tx,Ti3C2Tx,LiF/HCl minimally intensive layer delamination...,PVA/H2SO4 gel electrolyte,None
357,zhang2018.pdf,I-Ti3C2Tx (≈100 nm),Ti3C2Tx,LiF/HCl minimally intensive layer delamination...,PVA/H2SO4 gel electrolyte,None


In [28]:
# Save passed corpus only as csv
passed_csv = output_dir() / "h2so4_corpus_passed.csv"
passed.to_csv(passed_csv, index=False)
print(f"\nwrote {passed_csv} (only PASS rows)")


wrote terra\h2so4_corpus_passed.csv (only PASS rows)


### 4. Corpus overview

Where the data actually is, and where it isn't. The coverage table is the one to read
carefully — a descriptor populated in 15% of electrodes is not usable as a model
feature no matter how much you want it.

In [29]:
# Electrodes per paper — outliers usually mean a splitting mistake
per_paper = corpus.groupby("source_file").size().sort_values(ascending=False)
print("electrodes per paper:")
print(f"   median {per_paper.median():.0f}   max {per_paper.max()}   min {per_paper.min()}")
print("\nmost electrodes:")
print(per_paper.head(5).to_string())
if (per_paper > 10).any():
    print("\n  papers with >10 electrodes — check these for over-splitting:")
    print("  " + ", ".join(per_paper[per_paper > 10].index[:5]))

electrodes per paper:
   median 5   max 17   min 1

most electrodes:
source_file
Advanced Materials - 2019 - Tian - Multifunctional Nanocomposites with High Strength and Capacitance Using 2D MXene and 1D.pdf    17
d6ta00010j.pdf                                                                                                                    15
ma2022.pdf                                                                                                                        11
c8dt04374d.pdf                                                                                                                    10
nenergy2017105.pdf                                                                                                                 9

  papers with >10 electrodes — check these for over-splitting:
  Advanced Materials - 2019 - Tian - Multifunctional Nanocomposites with High Strength and Capacitance Using 2D MXene and 1D.pdf, d6ta00010j.pdf, ma2022.pdf


In [30]:
# Field coverage: how often is each descriptor actually reported?
value_cols = [c for c in corpus.columns
              if not c.endswith(("__unit", "__conf"))
              and c not in ("source_file", "source_path", "electrode_label", "role",
                            "is_primary", "doi", "year", "authors")]

cov = pd.DataFrame({
    "field": value_cols,
    "filled": [corpus[c].notna().sum() for c in value_cols],
    "pct": [round(100 * corpus[c].notna().mean(), 1) for c in value_cols],
}).sort_values("pct", ascending=False)
cov["usable"] = cov["pct"].apply(lambda p: "yes" if p >= 60 else
                                 ("marginal" if p >= 30 else "too sparse"))
print(f"coverage across {len(corpus)} electrodes:\n")
print(cov.to_string(index=False))

coverage across 366 electrodes:

                  field  filled   pct     usable
            composition     366 100.0        yes
       synthesis_method     362  98.9        yes
          mxene_formula     344  94.0        yes
            electrolyte     343  93.7        yes
               layer_no     300  82.0        yes
gravimetric_capacitance     165  45.1   marginal
             flake_size     163  44.5   marginal
              scan_rate     137  37.4   marginal
     interlayer_spacing     122  33.3   marginal
    electrode_thickness     120  32.8   marginal
        current_density      82  22.4 too sparse
 volumetric_capacitance      82  22.4 too sparse
           mass_loading      74  20.2 too sparse
      areal_capacitance      58  15.8 too sparse
                    ssa      51  13.9 too sparse
          pore_diameter      45  12.3 too sparse
               porosity      11   3.0 too sparse
             tortuosity       0   0.0 too sparse


In [31]:
# Confidence breakdown — how much was measured per-sample vs inherited from methods?
print("stated / derived / uncertain, by field:\n")
for c in value_cols:
    conf_col = c + "__conf"
    if conf_col not in corpus.columns:
        continue
    vc = corpus[conf_col].value_counts()
    tot = vc.sum()
    if tot == 0:
        continue
    s, d, u = vc.get("stated", 0), vc.get("derived", 0), vc.get("uncertain", 0)
    warn = "   <-- mostly inherited" if d > s else ""
    print(f"   {c:26s} {s:4d} / {d:4d} / {u:4d}{warn}")

stated / derived / uncertain, by field:

   mxene_formula               340 /   22 /    4
   composition                 349 /   16 /    1
   synthesis_method            284 /   81 /    1
   layer_no                    165 /  186 /   15   <-- mostly inherited
   interlayer_spacing          254 /   31 /   81
   flake_size                  176 /  134 /   56
   porosity                    257 /    0 /  109
   pore_diameter               240 /   18 /  108
   tortuosity                  262 /    0 /  104
   electrode_thickness         265 /   21 /   80
   mass_loading                221 /   40 /  105
   electrolyte                  58 /  292 /   16   <-- mostly inherited
   ssa                         263 /    2 /  101
   scan_rate                   277 /    0 /   89
   current_density             249 /    7 /  110
   gravimetric_capacitance     291 /    0 /   75
   volumetric_capacitance      269 /    0 /   97
   areal_capacitance           277 /    0 /   89


### 5. Papers that need attention

Two buckets. Failures are usually scanned PDFs with no text layer — run them through
OCR and rerun the batch (cached papers are skipped, so only the fixed ones cost money).
Flagged papers extracted successfully but tripped a consistency check.

In [32]:
if failures:
    print(f"{len(failures)} failed:\n")
    for name, err in failures:
        print(f"   {name}\n      {err}")
    print("\nFor scanned PDFs:  ocrmypdf in.pdf out.pdf   then rerun batch_extract")
else:
    print("no failures")

if flagged:
    print(f"\n{len(flagged)} flagged:\n")
    for name, ws in flagged:
        print(f"   {name}")
        for w in ws:
            print(f"      ! {w}")
else:
    print("\nnothing flagged")

1 failed:

   s40820-020-00450-0.pdf
      Invalid object in /Pages

For scanned PDFs:  ocrmypdf in.pdf out.pdf   then rerun batch_extract

20 flagged:

   1-s2.0-S2369969821000785-main.pdf
      ! 'CPCM' has no capacitance of any kind
   1-s2.0-S2405829723005251-main.pdf
      ! 'AMX-8D-500' has no capacitance of any kind
      ! 'AMX-8D-200' has no capacitance of any kind
      ! 'AMX-8D-100' has no capacitance of any kind
      ! 'AMX-12D-200' has no capacitance of any kind
   201807260933092014.pdf
      ! 'Ti3C2Tx film (1.0 ± 0.1 µm)' has no capacitance of any kind
   acsnano.9b10066.pdf
      ! 'MXene/RGO composite film (25 wt% RGO)' has no capacitance of any kind
      ! 'MXene/RGO composite film (75 wt% RGO)' has no capacitance of any kind
   Advanced Materials - 2023 - Arslanoglu - 3D Assembly of MXene Networks using a Ceramic Backbone with Controlled Porosity.pdf
      ! 'MX-PS electrodes with 40% longitudinal porosity, 180 mg mL−1 MXene' has no capacitance of any kind
      

### 6. Inspect one paper's extraction against the PDF

This is the hand-validation loop. Pick a paper, print its electrodes, open the PDF
alongside, and check that the sample list and the capacitances match. Do this for ten
papers before trusting the corpus.

In [33]:
import random

check = random.choice(corpus["source_file"].unique())
# check = "your_paper.pdf"   # or name one explicitly

sub = corpus[corpus["source_file"] == check]
print(f"{check} — {len(sub)} electrodes\n")
print(sub[["electrode_label", "role", "is_primary", "mxene_formula",
           "synthesis_method", "gravimetric_capacitance", "current_density"]]
      .to_string(index=False))

rec = next(r for r in results if r.get("source_file") == check)
print("\nmodel summary:\n  " + rec["summary"])

1-s2.0-S0013468622000433-main.pdf — 4 electrodes

electrode_label      role  is_primary mxene_formula                                               synthesis_method gravimetric_capacitance current_density
            MP0   control       False       Ti3C2Tx LiF/HCl etching followed by delamination and vacuum filtration                    None            None
            MP2 composite       False       Ti3C2Tx   LiF/HCl etching, PANI physical mixing, and vacuum filtration                    None            None
            MP5   primary        True       Ti3C2Tx   LiF/HCl etching, PANI physical mixing, and vacuum filtration                   425.7            None
            MP8 composite       False       Ti3C2Tx   LiF/HCl etching, PANI physical mixing, and vacuum filtration                    None            None

model summary:
  This study fabricated four freestanding Ti3C2Tx-based vacuum-filtered films: pure MXene (MP0) and PANI-nanofiber composites MP2, MP5, and MP8, differentiated

---

## Working with the corpus

**Filter on confidence before you model anything.** With multiple electrodes per paper,
`derived` now carries a specific meaning: the value was inherited from a shared methods
section rather than measured for that sample. That is often fine for electrolyte, and
often wrong for mass loading. Check per field:

```python
corpus["mass_loading__conf"].value_counts()
clean = corpus[corpus["gravimetric_capacitance__conf"] == "stated"]
```

**Watch for per-paper clustering.** A paper contributing eight electrodes has eight
times the influence of one contributing a single electrode, and electrodes within a
paper share a lab, a measurement rig, and an operator. If you fit a model on this,
group by `source_file` when you cross-validate rather than splitting rows at random —
otherwise you leak information between folds and overstate accuracy.

**The `role` column is the useful contrast.** Comparing `control` against `variant`
rows within the same paper isolates the effect of a treatment while holding the lab
constant, which is a much cleaner signal than comparing across groups.

**Parse units late.** The schema keeps `"412"` and `"F g-1"` in separate columns on
purpose. Write one conversion function per field once you know what you need. Trusting
the model to normalize during extraction is where silent factor-of-ten errors get in.

## Cost

Multi-electrode extraction costs more than single because output scales with electrode
count — input is unchanged. A paper with four electrodes runs roughly 3–4× the output
tokens of the single-electrode version, but yields four data rows instead of one, so
cost *per row* usually drops.

Rough figures at GPT-5.4 rates on a 15k-token paper: ~$0.06 for one electrode, ~$0.12
for four. Switch `MODEL` to `gpt-5.6-luna` for roughly 60% off, and use the Batch API
for another 50% on top if you can wait 24 hours.